In [0]:
%sql

-- FACT TRIP TABLE --
-- Grain: 1 row per individual trip (resolved at the borough, date, and hour level)

CREATE OR REPLACE TABLE nyc_mobility.mart.fact_trip AS
SELECT 
    t.trip_id,
    t.PULocationID  AS pu_location_id,
    t.DOLocationID  AS do_location_id,
    CAST(date_format(t.lpep_pickup_datetime, 'yyyyMMddHH') AS INT) AS pickup_date_hour_key,
    t.lpep_pickup_datetime  AS pickup_datetime,
    t.lpep_dropoff_datetime AS dropoff_datetime,
    HOUR(t.lpep_pickup_datetime)  AS pickup_hour,
    HOUR(t.lpep_dropoff_datetime) AS dropoff_hour,
    t.passenger_count,
    t.trip_distance,
    ROUND(CAST((UNIX_TIMESTAMP(t.lpep_dropoff_datetime) - UNIX_TIMESTAMP(t.lpep_pickup_datetime)) / 60.0 AS DOUBLE), 2) AS trip_duration_min,
    t.fare_amount,
    t.total_amount
FROM nyc_mobility.clean.green_taxi t
WHERE t.lpep_pickup_datetime IS NOT NULL
  AND t.lpep_dropoff_datetime IS NOT NULL
  AND (UNIX_TIMESTAMP(t.lpep_dropoff_datetime) - UNIX_TIMESTAMP(t.lpep_pickup_datetime)) / 60.0 BETWEEN 1.0 AND 1440.0;